# Sequential Chains — Composing Agents Step-by-Step

---

This notebook covers everything you need to know about **Chain** — Multigen's sequential
execution primitive. You will learn how to:

- Construct a multi-step chain and inspect its results
- Use conditional steps (`when=`) to skip steps based on runtime context
- Apply per-step timeouts
- Compose chains with the `|` and `+` operators
- Add middleware (logging, tracing, custom timing)
- Use the `Pipeline` alias for callable-style invocation
- Handle errors and inspect partial results
- Build a realistic 4-step document processing pipeline

## 1. What Is a Chain? The Sequential Execution Model

A `Chain` is an **ordered list of agents** that execute one after another, where each step
receives the full accumulated context from all previous steps.

```
Context accumulation diagram:

Initial ctx: {"text": "Hello World"}
      │
      ▼
┌─────────────┐
│  step_1     │  → returns {"tokens": ["Hello", "World"]}
└─────────────┘
      │  ctx now: {"text": "Hello World", "tokens": ["Hello", "World"]}
      ▼
┌─────────────┐
│  step_2     │  → returns {"upper_tokens": ["HELLO", "WORLD"]}
└─────────────┘
      │  ctx now: {"text": ..., "tokens": [...], "upper_tokens": [...]}
      ▼
┌─────────────┐
│  step_3     │  → returns {"result": "HELLO WORLD"}
└─────────────┘
      │
      ▼
ChainResult: final_output = {"result": "HELLO WORLD"}, full_context = {...all keys...}
```

### Key properties

- **Context is additive**: later steps see everything earlier steps wrote. Nothing is deleted
  unless an agent explicitly overwrites a key.
- **`final_output`**: the delta returned by the last step.
- **`full_context`**: the complete accumulated context after all steps.
- **Each step's output is stored**: you can retrieve `result["step_name"]` to inspect
  intermediate outputs.

In [ ]:
import asyncio
import time

from multigen import (
    Chain,
    Pipeline,
    FunctionAgent,
    logging_middleware,
    tracing_middleware,
    agent,
)

print("Multigen Chain imports successful.")

## 2. Basic 3-Step Chain — Tokenise → Uppercase → Join

Let's build the simplest possible chain: three agents that progressively transform a text string.

The three steps are:
1. **tokenise** — split the input text into a list of word tokens
2. **uppercase** — convert each token to uppercase
3. **join** — join the uppercase tokens back into a single string

Each agent is independently testable and reusable. The chain wires them together.

In [ ]:
# Step 1: tokenise — split text into word list
@agent
async def tokenise(ctx: dict) -> dict:
    """Split ctx['text'] on whitespace to produce ctx['tokens']."""
    return {"tokens": ctx.get("text", "").split()}

# Step 2: uppercase — transform each token to uppercase
@agent
async def uppercase_tokens(ctx: dict) -> dict:
    """Uppercase each token in ctx['tokens']."""
    return {"tokens": [t.upper() for t in ctx.get("tokens", [])]}

# Step 3: join — recombine tokens into a string
@agent
async def join_tokens(ctx: dict) -> dict:
    """Join ctx['tokens'] back into ctx['result']."""
    return {"result": " ".join(ctx.get("tokens", []))}

# Build the chain — steps execute in the order given
basic_chain = Chain(
    steps=[tokenise, uppercase_tokens, join_tokens],
    name="text_uppercase_pipeline",
)

# Run the chain
chain_result = await basic_chain.run({"text": "the quick brown fox"})

print("final_output :", chain_result.final_output)
# Expected: {'result': 'THE QUICK BROWN FOX'}

## 3. Accessing Step Outputs — `result.final_output`, `result["step_name"]`, `result.step_names`

A `ChainResult` stores not just the final output but every intermediate step's output.
This is critical for:

- **Debugging**: understanding exactly what each step produced
- **Branching logic**: downstream code that needs to inspect intermediate state
- **Auditing**: recording what each step did for compliance or logging

Access patterns:

| Expression | Returns |
|---|---|
| `result.final_output` | The delta dict from the last step |
| `result.full_context` | The complete merged context after all steps |
| `result["step_name"]` | The delta dict from the named step |
| `result.step_names` | Ordered list of step names that ran |
| `result.succeeded` | True if all steps completed without error |

In [ ]:
# Run the chain again to get a fresh ChainResult
chain_result = await basic_chain.run({"text": "hello world from multigen"})

# Access patterns
print("── final_output ──────────────────────────")
print(chain_result.final_output)               # last step's delta

print("\n── full_context ──────────────────────────")
print(chain_result.full_context)               # everything accumulated

print("\n── step_names ────────────────────────────")
print(chain_result.step_names)                 # ['tokenise', 'uppercase_tokens', 'join_tokens']

print("\n── result['tokenise'] ────────────────────")
print(chain_result["tokenise"])                # {'tokens': ['hello', 'world', 'from', 'multigen']}

print("\n── result['uppercase_tokens'] ────────────")
print(chain_result["uppercase_tokens"])        # {'tokens': ['HELLO', 'WORLD', 'FROM', 'MULTIGEN']}

print("\n── succeeded ─────────────────────────────")
print(chain_result.succeeded)                  # True

## 4. ChainResult Inspection — All Fields

Let's inspect the complete `ChainResult` structure to understand every field available.

In [ ]:
# Inspect all fields on ChainResult
cr = chain_result  # alias for brevity

print("Type             :", type(cr).__name__)
print("succeeded        :", cr.succeeded)         # bool — True if no errors
print("step_names       :", cr.step_names)         # list[str] — ordered step names
print("steps_completed  :", cr.steps_completed)    # int — number of steps that ran
print("total_ms         :", f"{cr.total_ms:.2f} ms")  # total wall-clock time
print("errors           :", cr.errors)             # dict[step_name, Exception] — usually {}
print("final_output     :", cr.final_output)       # last step delta
print("full_context keys:", list(cr.full_context.keys()))  # all accumulated keys

# Per-step timing (available when tracing middleware is active)
# print("step_timings     :", cr.step_timings)  # dict[step_name, float_ms]

## 5. Conditional Steps — `when=` Predicate

The `when=` parameter on a chain step accepts a callable `(ctx: dict) -> bool`. When the
predicate returns `False`, the step is **skipped entirely**: the agent is not called, no
delta is produced, and the context passes through unchanged.

Skipped steps still appear in `result.step_names` but with a `(skipped)` annotation, so
you can see exactly which steps ran and which were bypassed.

### Why prefer `when=` over putting `if` inside the agent?

- The condition is **visible at the pipeline level**, not buried in agent logic
- The agent remains **pure and focused** — it does one thing and assumes the inputs are valid
- Observability tools can emit `step_skipped` events without instrumenting agent code
- You can change routing logic without touching the agent implementation

In [ ]:
# A chain where step 3 (profanity filter) only runs if text is flagged as unsafe
@agent
async def safety_check(ctx: dict) -> dict:
    """Simple mock safety checker. Sets 'needs_filter' flag."""
    text = ctx.get("text", "")
    bad_words = ["spam", "scam", "hack"]
    return {"needs_filter": any(w in text.lower() for w in bad_words)}

@agent
async def apply_filter(ctx: dict) -> dict:
    """Replace flagged words. Only runs when needs_filter is True."""
    bad_words = ["spam", "scam", "hack"]
    text = ctx.get("text", "")
    for w in bad_words:
        text = text.replace(w, "[REMOVED]")
    return {"text": text, "was_filtered": True}

@agent
async def log_result(ctx: dict) -> dict:
    """Append a status message to the output."""
    status = "filtered" if ctx.get("was_filtered") else "clean"
    return {"status": status}

# Build a chain with a conditional step
content_chain = Chain(
    steps=[
        safety_check,
        # apply_filter only runs when safety_check set needs_filter=True
        (apply_filter, {"when": lambda ctx: ctx.get("needs_filter", False)}),
        log_result,
    ],
    name="content_moderation",
)

# Clean input — apply_filter should be skipped
r1 = await content_chain.run({"text": "A perfectly normal sentence."})
print("Clean input:")
print("  final_output :", r1.final_output)
print("  steps_completed:", r1.steps_completed)
print()

# Unsafe input — apply_filter should run
r2 = await content_chain.run({"text": "This is spam! Click here to hack accounts."})
print("Unsafe input:")
print("  text after filter:", r2.full_context.get("text"))
print("  was_filtered      :", r2.full_context.get("was_filtered"))
print("  final_output      :", r2.final_output)

## 6. Step Timeout — `timeout=5.0`

Each step can have an independent timeout (in seconds). If the step does not complete within
the timeout window, it raises a `TimeoutError` (or is treated as an error, depending on the
chain's `fail_fast` setting).

Timeouts are essential for:
- **Preventing pipeline hangs** when an external service is slow
- **SLA enforcement** — each step has a budget and must respect it
- **Graceful degradation** — with `fail_fast=False`, a timed-out step is skipped and
  the pipeline continues with partial results

Timeout is specified via the step options dict: `(agent, {"timeout": seconds})`

In [ ]:
# An agent that sleeps for a configurable duration
@agent
async def slow_enrich(ctx: dict) -> dict:
    """Simulates a slow external enrichment API (200 ms)."""
    await asyncio.sleep(0.2)  # 200 ms — intentionally slow
    return {"enriched": True, "extra_data": "enrichment_payload"}

@agent
async def fast_step(ctx: dict) -> dict:
    """A fast step that always completes quickly."""
    return {"fast_done": True}

# Chain with a generous timeout (step will succeed)
chain_ok = Chain(
    steps=[
        fast_step,
        (slow_enrich, {"timeout": 1.0}),   # 1 second timeout — plenty of time
    ],
    name="chain_with_generous_timeout",
)

r = await chain_ok.run({"input": "data"})
print("Generous timeout — succeeded:", r.succeeded)
print("enriched:", r.full_context.get("enriched"))

In [ ]:
# Chain with a tight timeout (step will fail)
chain_timeout = Chain(
    steps=[
        fast_step,
        (slow_enrich, {"timeout": 0.05}),  # 50 ms timeout — slow_enrich takes 200 ms → timeout
    ],
    name="chain_with_tight_timeout",
    fail_fast=False,  # don't abort; continue with partial results
)

r = await chain_timeout.run({"input": "data"})
print("Tight timeout — succeeded:", r.succeeded)       # False
print("errors          :", r.errors)                    # {'slow_enrich': TimeoutError(...)}
print("fast_done present:", "fast_done" in r.full_context)  # True — fast_step ran fine
print("enriched present :", "enriched" in r.full_context)   # False — slow_enrich timed out

## 7. Operator Syntax — `chain_a | chain_b` and `chain + agent`

Chains support two composition operators for ergonomic pipeline construction:

| Operator | Meaning |
|---|---|
| `chain_a \| chain_b` | Concatenate two chains into a new chain (all steps of a, then all steps of b) |
| `chain + agent` | Append a single agent as a new step at the end of the chain |
| `agent + chain` | Prepend a single agent at the start of the chain |

These operators always produce a **new Chain** object — they do not mutate the originals.
This makes them safe to use in functional-style compositions.

They are useful when you want to **reuse sub-chains** across different pipelines.
For example, a normalisation chain might be prepended to many different downstream chains.

In [ ]:
# Sub-chain A: text normalisation
@agent
async def strip_whitespace(ctx: dict) -> dict:
    return {"text": ctx.get("text", "").strip()}

@agent
async def to_lowercase(ctx: dict) -> dict:
    return {"text": ctx.get("text", "").lower()}

normalisation_chain = Chain(
    steps=[strip_whitespace, to_lowercase],
    name="normalisation",
)

# Sub-chain B: feature extraction
@agent
async def count_words(ctx: dict) -> dict:
    return {"word_count": len(ctx.get("text", "").split())}

@agent
async def count_chars(ctx: dict) -> dict:
    return {"char_count": len(ctx.get("text", ""))}

feature_chain = Chain(
    steps=[count_words, count_chars],
    name="features",
)

# Concatenate using | operator — produces a 4-step chain
full_pipeline = normalisation_chain | feature_chain
print("Combined chain steps:", full_pipeline.step_names)
# Expected: ['strip_whitespace', 'to_lowercase', 'count_words', 'count_chars']

r = await full_pipeline.run({"text": "  Hello World from Multigen  "})
print("word_count:", r.full_context["word_count"])
print("char_count:", r.full_context["char_count"])
print("normalised:", r.full_context["text"])

In [ ]:
# Append a single agent using the + operator
@agent
async def summarise_stats(ctx: dict) -> dict:
    """Create a summary string from word and char counts."""
    return {"stats_summary": f"{ctx.get('word_count',0)} words, {ctx.get('char_count',0)} chars"}

extended_pipeline = full_pipeline + summarise_stats
print("Extended pipeline steps:", extended_pipeline.step_names)

r2 = await extended_pipeline.run({"text": "  Operator syntax is elegant  "})
print("stats_summary:", r2.full_context["stats_summary"])

## 8. Middleware — Built-in `logging_middleware` and `tracing_middleware`

Middleware wraps the execution of each step, intercepting calls **before** and **after**
the agent runs. Multigen ships two ready-made middleware functions:

- **`logging_middleware`**: prints a log line for each step start/end with the step name
  and whether it succeeded
- **`tracing_middleware`**: records per-step timing (available in `result.step_timings`)
  and emits structured trace events

Middleware is passed as a list to the `Chain` constructor. Multiple middleware stack
in the order given (first in list = outermost wrapper).

```python
Chain(
    steps=[...],
    middleware=[logging_middleware, tracing_middleware],
)
```

In [ ]:
# Chain with both built-in middleware enabled
@agent
async def step_alpha(ctx: dict) -> dict:
    await asyncio.sleep(0.01)
    return {"alpha": ctx.get("value", 0) * 2}

@agent
async def step_beta(ctx: dict) -> dict:
    await asyncio.sleep(0.02)
    return {"beta": ctx.get("alpha", 0) + 10}

@agent
async def step_gamma(ctx: dict) -> dict:
    await asyncio.sleep(0.01)
    return {"gamma": ctx.get("beta", 0) ** 2}

traced_chain = Chain(
    steps=[step_alpha, step_beta, step_gamma],
    middleware=[logging_middleware, tracing_middleware],
    name="traced_computation",
)

r = await traced_chain.run({"value": 5})
# logging_middleware will print lines like:
#   [LOG] step_alpha  START
#   [LOG] step_alpha  END (succeeded, 10ms)
#   [LOG] step_beta   START ...

print("\n── Results ────────────────────────")
print("alpha :", r.full_context.get("alpha"))   # 10
print("beta  :", r.full_context.get("beta"))    # 20
print("gamma :", r.full_context.get("gamma"))   # 400

# tracing_middleware populates step_timings
if hasattr(r, "step_timings") and r.step_timings:
    print("\n── Step timings ───────────────────")
    for name, ms in r.step_timings.items():
        print(f"  {name}: {ms:.1f} ms")

## 9. Custom Middleware — Writing a Timing Middleware from Scratch

Middleware follows a simple convention: it is a higher-order function that wraps a step.

```python
def my_middleware(next_fn):
    async def wrapper(ctx: dict) -> dict:
        # do something BEFORE
        result = await next_fn(ctx)
        # do something AFTER
        return result
    return wrapper
```

The `next_fn` is the actual step function (or the next middleware's wrapper). You can
inspect `ctx`, modify `ctx` before passing it on, modify the result before returning it,
or raise exceptions to abort the step.

This design is identical to middleware in WSGI, Django, or Express.js — the "onion" model.

In [ ]:
# Custom timing middleware — records wall-clock time for every step
def timing_middleware(next_fn):
    """
    Wraps each step with a high-resolution timer.
    Appends {'__timing__': {step_name: elapsed_ms}} to the step's output.
    """
    async def wrapper(ctx: dict) -> dict:
        step_name = getattr(next_fn, '__name__', 'unknown')
        t0 = time.perf_counter()
        result = await next_fn(ctx)   # call the actual step
        elapsed_ms = (time.perf_counter() - t0) * 1000
        # Merge timing info into the step result (stored in ChainResult)
        timings = ctx.get("__timings__", {})
        timings[step_name] = round(elapsed_ms, 2)
        return {**result, "__timings__": timings}
    return wrapper

# Custom audit middleware — logs every input/output pair
def audit_middleware(next_fn):
    """Prints the keys added by each step for an audit trail."""
    async def wrapper(ctx: dict) -> dict:
        keys_before = set(ctx.keys())
        result = await next_fn(ctx)
        new_keys = set(result.keys()) - keys_before - {"__timings__"}
        step_name = getattr(next_fn, '__name__', 'unknown')
        if new_keys:
            print(f"[AUDIT] {step_name} added keys: {sorted(new_keys)}")
        return result
    return wrapper

# Build a chain using both custom middleware
instrumented_chain = Chain(
    steps=[tokenise, uppercase_tokens, join_tokens],
    middleware=[timing_middleware, audit_middleware],
    name="instrumented",
)

r = await instrumented_chain.run({"text": "the quick brown fox"})
print("\nStep timings recorded:", r.full_context.get("__timings__"))
print("Final result:", r.final_output)

## 10. Pipeline Alias — Callable Interface

`Pipeline` is an alias for `Chain` that emphasises the **callable interface**.
While `Chain.run(ctx)` is the primary API, `Pipeline` is used when you want to treat
the whole pipeline as a single callable function:

```python
result = await pipeline(ctx)   # __call__ delegates to run()
```

This is especially useful when passing a pipeline as a callback to another system,
or when building higher-order abstractions that expect a `callable(ctx) -> result` interface.

`Pipeline` and `Chain` are fully interchangeable — the difference is purely stylistic.

In [ ]:
# Pipeline is identical to Chain but uses __call__ style
text_pipeline = Pipeline(
    steps=[strip_whitespace, to_lowercase, count_words],
    name="text_pipeline",
)

# Callable interface — same as await text_pipeline.run(...)
r1 = await text_pipeline({"text": "  Hello World  "})
print("Via __call__:", r1.full_context["word_count"])  # 2

# Run interface — identical result
r2 = await text_pipeline.run({"text": "  Hello World  "})
print("Via .run():  ", r2.full_context["word_count"])  # 2

# Pipeline can be passed as a function to other code
async def process_batch(items: list, pipeline) -> list:
    """Process multiple texts through a pipeline."""
    return [await pipeline({"text": item}) for item in items]

batch_results = await process_batch(
    ["  foo bar  ", "  baz qux quux  ", "  one  "],
    text_pipeline
)
for r in batch_results:
    print(r.full_context["word_count"], "words:", r.full_context["text"])

## 11. Error Handling — What Happens When a Step Raises?

Chains have two error modes controlled by `fail_fast`:

| `fail_fast` value | Behaviour on step error |
|---|---|
| `True` (default) | Stop immediately, raise the exception, partial ChainResult available |
| `False` | Record the error, skip the step, continue with remaining steps |

With `fail_fast=False`, the chain **always completes** but may have gaps in the context.
Check `result.errors` (a dict mapping step name to exception) to see what failed.

This is useful for **best-effort enrichment pipelines** where partial results are better
than no results.

In [ ]:
# A step that randomly fails
@agent
async def flaky_enricher(ctx: dict) -> dict:
    """Simulates an unreliable external enrichment service."""
    if ctx.get("force_fail"):
        raise ValueError("Enrichment service unavailable (simulated failure)")
    return {"enriched_data": "premium_enrichment_payload"}

@agent
async def final_formatter(ctx: dict) -> dict:
    """Always runs — formats whatever data is available."""
    base = ctx.get("text", "")
    extra = ctx.get("enriched_data", "(enrichment unavailable)")
    return {"formatted": f"{base} | {extra}"}

resilient_chain = Chain(
    steps=[tokenise, flaky_enricher, final_formatter],
    fail_fast=False,   # continue even if flaky_enricher fails
    name="resilient",
)

# Run 1: normal execution
r_ok = await resilient_chain.run({"text": "hello world", "force_fail": False})
print("Normal run:")
print("  succeeded:", r_ok.succeeded)
print("  formatted:", r_ok.full_context.get("formatted"))
print()

# Run 2: flaky step fails, pipeline continues, partial result returned
r_fail = await resilient_chain.run({"text": "hello world", "force_fail": True})
print("Failed step run (fail_fast=False):")
print("  succeeded :", r_fail.succeeded)   # False
print("  errors    :", {k: type(v).__name__ for k, v in r_fail.errors.items()})
print("  formatted :", r_fail.full_context.get("formatted"))  # still populated by final_formatter

In [ ]:
# With fail_fast=True (the default), the chain stops at the first error
strict_chain = Chain(
    steps=[tokenise, flaky_enricher, final_formatter],
    fail_fast=True,
    name="strict",
)

try:
    r_strict = await strict_chain.run({"text": "hello world", "force_fail": True})
except ValueError as e:
    print(f"Chain raised ValueError: {e}")
    print("  final_formatter did NOT run (fail_fast stopped execution)")

## 12. Real-World Mini-Example: 4-Step Document Processing Pipeline

Let's put everything together in a realistic scenario: processing an incoming document
through four stages:

```
load  →  clean  →  summarise  →  tag
 ↓          ↓          ↓           ↓
raw_text  cleaned_text  summary   tags[]
```

Each step enriches the context with new keys. The final context contains all intermediate
and final outputs, which can be stored in a database or returned to the caller.

We include:
- A conditional step (`tag` only runs if the summary is non-empty)
- Per-step timeouts
- Tracing middleware to record latency
- `fail_fast=False` for resilience

In [ ]:
# Step 1: Load — simulate fetching a document from storage
@agent
async def load_document(ctx: dict) -> dict:
    """Fetch document from storage (mocked). Reads ctx['doc_id']."""
    await asyncio.sleep(0.02)  # simulate DB read
    doc_id = ctx.get("doc_id", "unknown")
    raw_text = (
        "  Artificial Intelligence (AI) is intelligence demonstrated by machines, "
        "as opposed to the natural intelligence displayed by animals and humans. "
        "AI research has been defined as the field of study of intelligent agents. "
        "Applications include natural language processing, computer vision, and robotics.  "
    )
    return {"raw_text": raw_text, "doc_id": doc_id, "loaded": True}

# Step 2: Clean — normalise whitespace, remove special characters
@agent
async def clean_document(ctx: dict) -> dict:
    """Normalise text: strip, collapse whitespace."""
    raw = ctx.get("raw_text", "")
    import re
    cleaned = re.sub(r"\s+", " ", raw).strip()
    return {"text": cleaned}

# Step 3: Summarise — mock LLM summarisation
@agent
async def summarise_document(ctx: dict) -> dict:
    """Mock summarisation: return first sentence."""
    await asyncio.sleep(0.05)  # simulate LLM latency
    text = ctx.get("text", "")
    first_sentence = text.split(".")[0].strip() + "." if "." in text else text
    return {"summary": first_sentence}

# Step 4: Tag — extract topic tags (only runs if summary exists)
@agent
async def tag_document(ctx: dict) -> dict:
    """Mock keyword tagging from summary."""
    await asyncio.sleep(0.03)
    summary = ctx.get("summary", "")
    stopwords = {"is", "a", "the", "of", "as", "by", "to", "and", "in", "an"}
    words = [w.strip(",.").lower() for w in summary.split()]
    tags = list({w for w in words if w not in stopwords and len(w) > 3})[:5]
    return {"tags": sorted(tags), "tagged": True}

# Assemble the full 4-step document pipeline
doc_pipeline = Chain(
    steps=[
        (load_document,     {"timeout": 5.0}),
        (clean_document,    {"timeout": 2.0}),
        (summarise_document,{"timeout": 10.0}),
        # Only tag if we got a non-empty summary
        (tag_document, {"timeout": 5.0, "when": lambda ctx: bool(ctx.get("summary"))}),
    ],
    middleware=[tracing_middleware],
    fail_fast=False,
    name="document_processing_pipeline",
)

result = await doc_pipeline.run({"doc_id": "doc_42"})

print("Pipeline succeeded:", result.succeeded)
print("Steps completed:", result.steps_completed)
print()
print("raw_text (first 60 chars):", result.full_context.get("raw_text", "")[:60])
print("text     (first 60 chars):", result.full_context.get("text", "")[:60])
print("summary                  :", result.full_context.get("summary"))
print("tags                     :", result.full_context.get("tags"))

if hasattr(result, "step_timings") and result.step_timings:
    print()
    print("── Step timings ───────────────────────────")
    for step, ms in result.step_timings.items():
        print(f"  {step:25s}: {ms:.1f} ms")

## Summary

You have now learned every major feature of Multigen's `Chain`:

| Feature | How to use it |
|---|---|
| Basic chain | `Chain(steps=[a, b, c])` |
| Step output | `result["step_name"]`, `result.final_output` |
| Conditional step | `(agent, {"when": lambda ctx: ...})` |
| Step timeout | `(agent, {"timeout": 5.0})` |
| Concatenation | `chain_a \| chain_b` |
| Append step | `chain + agent` |
| Built-in middleware | `middleware=[logging_middleware, tracing_middleware]` |
| Custom middleware | `def my_mw(next_fn): async def wrapper(ctx): ...` |
| Pipeline alias | `Pipeline(steps=[...])`, callable via `await pipeline(ctx)` |
| Error resilience | `fail_fast=False`, check `result.errors` |

**Next notebook**: `14_parallel_execution.ipynb` — fan-out, race conditions, map-reduce,
and batch processing with Multigen's parallel primitives.